In [1]:
import os
import pandas as pd
import numpy as np
from Bio import SeqIO
import pandas as pd
import re
import os

In [2]:
def parse_drug(file):
    with open(file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        lines = lines[27:]  # 跳过前面27行注释
        
        content = ''.join(lines)
        
        drug_list = []
        
        for entry in content.split('\n\n')[2:]:
            if not entry.strip():
                continue
            
            details = {}
            for i, info in enumerate(entry.split('\n')):
                if not info.strip():
                    continue
                    
                _, name, item = info.split('\t')
                
                if i == 0 and name == 'DRUG__ID':
                    drug_id = item
                    details['DRUG_ID'] = item
                else:
                    # 使用setdefault避免重复覆盖
                    details.setdefault(name, item)
            
            if drug_id:
                drug_list.append(details)
        
        return pd.DataFrame(drug_list)

In [3]:
def parse_prot(file_path):
    """
    解析TTD药物靶点数据库文件，提取蛋白-药物对信息
    
    Parameters:
    file_path: str, 文件路径
    
    Returns:
    pd.DataFrame, 包含蛋白-药物对信息的DataFrame
    """
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 按空行分割，每个target为一个block
    blocks = content.split('\n\n')
    
    # 找到第一个真正的数据块（以TARGETID开头）
    data_blocks = []
    for block in blocks:
        if block.strip().startswith('T') and 'TARGETID' in block:
            data_blocks.append(block)
    
    # 存储所有蛋白-药物对
    records = []
    
    for block in data_blocks:
        lines = block.strip().split('\n')
        
        # 当前target的基本信息
        target_info = {}
        drug_info_list = []
        
        for line in lines:
            if not line.strip():
                continue
            
            parts = line.split('\t')
            if len(parts) < 3:
                continue
            
            target_id = parts[0]
            field_name = parts[1]
            field_value = '\t'.join(parts[2:])  # 处理值中可能包含的制表符
            
            if field_name == 'DRUGINFO':
                # 解析药物信息：DRUGINFO\tD0O6UY\tPemigatinib\tApproved
                drug_parts = field_value.split('\t')
                if len(drug_parts) >= 2:
                    drug_info = {
                        'DRUGID': drug_parts[0] if len(drug_parts) > 0 else '',
                        'DRUGNAME': drug_parts[1] if len(drug_parts) > 1 else '',
                        'CLINICAL_STATUS': drug_parts[2] if len(drug_parts) > 2 else ''
                    }
                    drug_info_list.append(drug_info)
            else:
                # 存储基本信息
                if field_name not in target_info:
                    target_info[field_name] = field_value
        
        # 为每个药物创建一个记录
        for drug_info in drug_info_list:
            record = {
                'TARGETID': target_info.get('TARGETID', ''),
                'FORMERID': target_info.get('FORMERID', ''),
                'UNIPROID': target_info.get('UNIPROID', ''),
                # 'SEQUENCE': target_info.get('SEQUENCE', ''),
                'TARGNAME': target_info.get('TARGNAME', ''),
                'GENENAME': target_info.get('GENENAME', ''),
                'TARGTYPE': target_info.get('TARGTYPE', ''),
                'BIOCLASS': target_info.get('BIOCLASS', ''),
                'ECNUMBER': target_info.get('ECNUMBER', ''),
                'DRUGID': drug_info['DRUGID'],
                'DRUGNAME': drug_info['DRUGNAME'],
                'CLINICAL_STATUS': drug_info['CLINICAL_STATUS'],
                'PDBSTRUC': target_info.get('PDBSTRUC', ''),
            }
            records.append(record)

    df = pd.DataFrame(records)
    df = df.reset_index(drop=True)
    return df

In [4]:
def parse_ttd_file(file_path):
    data = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if not line.startswith('D') or line.count('\t') < 2:
                continue
            parts = line.split('\t')
            if len(parts) != 3:
                continue
            drug_id, key, value = parts
            if drug_id not in data:
                data[drug_id] = {}
            if key != 'TTDDRUID':
                # 处理CAS字段，去掉前缀
                if key == 'CASNUMBE' and value.startswith('CAS '):
                    value = value[4:]
                data[drug_id][key] = value
    df = pd.DataFrame.from_dict(data, orient='index')
    df.index.name = 'TTDDRUID'
    df.reset_index(inplace=True)
    return df

In [5]:
from rdkit import Chem

def is_valid_smiles(smiles):
    if pd.isna(smiles):
        return False
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        Chem.SanitizeMol(mol)
        return True
    except:
        return False

In [6]:
save_dir = './process_data/'
os.makedirs(save_dir, exist_ok=True)

file_path = r"./raw_data/"
drug_file = os.path.join(file_path, 'P1-02-TTD_drug_download.txt')
drug_df = parse_drug(drug_file)
drug_df.columns =['DRUGID', 'TRADNAME', 'DRUGCOMP', 'THERCLAS', 'DRUGTYPE', 'DRUGINCH', 'INCHIKEY', 'SMILES', 'HIGHSTAT', 'COMPCLAS']
valid_drug_df = drug_df[drug_df['SMILES'].apply(is_valid_smiles)].copy()
valid_drug_df.to_excel(save_dir + 'DrugInfo.xlsx', index=False)

drug_prot_file = os.path.join(file_path, 'P1-01-TTD_target_download.txt')
drug_prot_df = parse_prot(drug_prot_file)
drug_prot_df.columns = ['TARGETID', 'FORMERID', 'ENTRYNAME', 'TARGNAME', 'GENENAME', 'TARGTYPE',
                        'BIOCLASS', 'ECNUMBER', 'DRUGID', 'DRUGNAME', 'CLINICAL_STATUS', 'PDBSTRUC']
drug_prot_df.to_excel(save_dir + 'DrugProtInfo.xlsx', index=False)

cross_match_file = os.path.join(file_path, 'P1-03-TTD_crossmatching.txt')
cross_match_df = parse_ttd_file(cross_match_file)
cross_match_df.to_excel(save_dir + 'CrossMatchInfo.xlsx', index=False)

[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] WARNING: not removing hydrogen atom without neighbors
[11:10:29] SMILES Parse Error: syntax error while parsing: failed
[11:10:29] SMILES Parse Error: check for mistakes around position 1:
[11:10:29] failed
[11:10:29] ^
[11:10:29] SMILES Parse Error: Failed parsing SMILES 'failed' for input: 'failed'
[11:10:29] WARNING: not removing hydr

In [7]:
uniprot_df = pd.read_excel(r'../UniprotKB/uniprotkb_2026_04_01.xlsx')
sub_uniprot_df = uniprot_df[['Entry', 'Entry Name', 'Sequence']]
sub_uniprot_df.columns = ['UNIPROTID', 'ENTRYNAME', 'SEQUENCE']

In [8]:
drug_df = pd.read_excel(r'./process_data/DrugInfo.xlsx')
sub_drug_df = drug_df[['DRUGID', 'INCHIKEY', 'SMILES']]

In [9]:
prot_drug_df = pd.read_excel(save_dir + 'DrugProtInfo.xlsx')
merge_prot_df = pd.merge(prot_drug_df, sub_uniprot_df, on='ENTRYNAME', how='inner')
merge_drug_df = pd.merge(merge_prot_df, drug_df, on='DRUGID', how='inner')

## 筛选出SMILE和蛋白序列均不为空的行
merge_drug_df = merge_drug_df[(merge_drug_df['SMILES'].notna()) & 
                              (merge_drug_df['SEQUENCE'].notna()) & 
                              (merge_drug_df['DRUGID'].notna()) ]

merge_drug_df = merge_drug_df.drop_duplicates(subset=['DRUGID', 'UNIPROTID']).reset_index(drop=True)
print('Before filtering, number of pairs:', len(merge_drug_df))

save_dir = './final_data/'
os.makedirs(save_dir, exist_ok=True)
merge_drug_df.to_excel(save_dir + 'TTD.xlsx', index=False)

print('After filtering, number of proteins:', len(merge_drug_df['UNIPROTID'].unique()))
print('After filtering, number of drugs:', len(merge_drug_df['DRUGID'].unique()))
print('After filtering, number of pairs:', len(merge_drug_df))

Before filtering, number of pairs: 28965
After filtering, number of proteins: 1614
After filtering, number of drugs: 19375
After filtering, number of pairs: 28965


In [10]:
import pandas as pd
import random
from itertools import product

# 读取数据
pair_df = pd.read_excel(r'./final_data/TTD.xlsx')
print('Before filtering, number of pairs:', len(pair_df))

# ===== 1. 构建字典 =====
drug_df = pair_df[['DRUGID', 'DRUGNAME', 'INCHIKEY', 'SMILES']].drop_duplicates()

protein_df = pair_df[['TARGETID', 'ENTRYNAME', 'GENENAME', 'TARGNAME', 'TARGTYPE', 'BIOCLASS', 'ECNUMBER', 
                      'UNIPROTID', 'SEQUENCE']].drop_duplicates()

drug_dict = dict(zip(drug_df['DRUGID'], drug_df['SMILES']))
protein_dict = dict(zip(protein_df['UNIPROTID'], protein_df['SEQUENCE']))

print('Number of drugs:', len(drug_dict))
print('Number of proteins:', len(protein_dict))

# 保存字典
drug_df.to_excel(r'./final_data/DrugInfo.xlsx', index=False)
protein_df.to_excel(r'./final_data/ProtInfo.xlsx', index=False)

# ===== 2. 正样本 =====
pos_pairs = set(zip(pair_df['DRUGID'], pair_df['UNIPROTID']))
num_pos = len(pos_pairs)
print('Number of positive pairs:', num_pos)

# ===== 3. 生成全部负样本 =====
all_drugs = pair_df['DRUGID'].dropna().unique()
all_proteins = pair_df['UNIPROTID'].dropna().unique()

print('Generating all candidate negative pairs...')
all_pairs = set(product(all_drugs, all_proteins))
all_neg_pairs = list(all_pairs - pos_pairs)

print('Total candidate negative pairs:', len(all_neg_pairs))

ratios = [1, 3, 5, 7, 10]
seeds = [1, 11, 111, 1111, 11111]

for ratio in ratios:
    print(f'Generating dataset with ratio 1:{ratio}...')
    save_dir = f'./final_data/1_{ratio}'
    os.makedirs(save_dir, exist_ok=True)

    num_neg = num_pos * ratio

    for i, seed in enumerate(seeds):
        print(f'Generating dataset {i+1} with seed {seed}...')
        random.seed(seed)
        sampled_neg = random.sample(all_neg_pairs, num_neg)

        neg_df = pd.DataFrame(sampled_neg, columns=['DRUGID', 'UNIPROTID'])
        neg_df['Label'] = 0

        pos_df = pd.DataFrame(list(pos_pairs), columns=['DRUGID', 'UNIPROTID'])
        pos_df['Label'] = 1

        dataset = pd.concat([pos_df, neg_df], ignore_index=True)
        dataset = dataset.sample(frac=1, random_state=seed).reset_index(drop=True)
        dataset.to_excel(os.path.join(save_dir, f'Set{i+1}.xlsx'), index=False)

print('All datasets generated!')

Before filtering, number of pairs: 28965
Number of drugs: 19375
Number of proteins: 1614
Number of positive pairs: 28965
Generating all candidate negative pairs...
Total candidate negative pairs: 31242285
Generating dataset with ratio 1:1...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:3...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:5...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:7...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Gene